# Aula 1 — Modelagem Matemática, Cinemática e Controle do Robô Otto

**Disciplina:** Introdução à Robótica e Sistemas Dinâmicos  
**Pré-requisitos:** Trigonometria básica, cálculo diferencial elementar  
**Ferramentas:** MATLAB, Simulink, Control System Toolbox

---

## Objetivos da aula

Ao final desta aula, o estudante será capaz de:

1. Descrever matematicamente a posição de uma perna articulada a partir dos ângulos das juntas (cinemática direta);
2. Modelar o movimento oscilatório das juntas por meio de funções senoidais;
3. Calcular velocidade e aceleração angular e relacioná-las ao torque exigido dos atuadores;
4. Compreender a estrutura de um sistema de controle em malha fechada com controlador PID;
5. Implementar e simular os modelos descritos no MATLAB e no Simulink.

---

## Motivação

Ao montar o robô Otto, você construiu um sistema físico com múltiplos atuadores, segmentos rígidos e restrições geométricas. Quando ele dá um passo, três fenômenos ocorrem simultaneamente:

- A **geometria** da estrutura se altera conforme os servos mudam de ângulo;
- A **física** impõe restrições de inércia e gravidade sobre os segmentos;
- O **controle** reage continuamente, corrigindo desvios entre a posição desejada e a posição real dos atuadores.

Esta aula conecta esses três domínios usando matemática elementar. O objetivo não é apenas fazer o robô andar, mas compreender quantitativamente por que ele se comporta da forma que se comporta — e diagnosticar o que ocorre quando algo falha.

### Relação com problemas de engenharia civil

Os princípios tratados aqui aparecem com frequência em engenharia civil:

| Conceito (robótica) | Análogo em engenharia civil |
|---|---|
| Cinemática de perna articulada | Geometria de treliça com barras rotacionadas |
| Oscilação senoidal da junta | Vibração livre sob carga harmônica |
| Centro de massa e equilíbrio do bípede | Centróide e condição de tombamento de estruturas |
| Torque gravitacional sobre segmento | Momento fletor por carga excêntrica |
| Controlador PID em malha fechada | Amortecedor ativo em estruturas sob ação sísmica |

A matemática subjacente é a mesma; o sistema físico é diferente.

---

## 1. Cinemática direta da perna

### 1.1 Formulação do problema

**Cinemática** descreve o movimento de corpos em termos de posição, velocidade e aceleração, sem considerar as causas do movimento. A **cinemática direta** resolve o seguinte problema:

> Dados os ângulos de cada junta, qual é a posição do efetuador final?

No Otto, o efetuador final é o pé. O problema é análogo a determinar a posição dos nós de uma treliça a partir das dimensões e orientações das barras.

### 1.2 Um segmento rotacionando

Considere um segmento rígido de comprimento $L$ fixado a um eixo de rotação na origem do sistema cartesiano. Quando o segmento forma um ângulo $\theta$ com o eixo horizontal, as coordenadas de sua extremidade livre são:

$$
x = L\cos(\theta), \qquad y = L\sin(\theta)
$$

O seno e o cosseno emergem naturalmente da decomposição de um vetor rotacionado sobre dois eixos ortogonais. Este resultado é o fundamento de toda a cinemática de sistemas articulados em cadeia aberta.

### 1.3 Dois segmentos em série: modelo da perna do Otto

A perna do Otto é modelada como dois segmentos rígidos conectados em série (comprimentos $L_1$ e $L_2$). O primeiro segmento gira com ângulo $\theta_1$ em relação ao referencial global; o segundo gira com ângulo $\theta_2$ em relação ao primeiro.

A posição da junta intermediária é:

$$
x_1 = L_1\cos(\theta_1), \qquad y_1 = L_1\sin(\theta_1)
$$

A posição do pé resulta da acumulação de rotações:

$$
\boxed{x_2 = L_1\cos(\theta_1) + L_2\cos(\theta_1 + \theta_2)}
$$

$$
\boxed{y_2 = L_1\sin(\theta_1) + L_2\sin(\theta_1 + \theta_2)}
$$

O ângulo acumulado $(\theta_1 + \theta_2)$ é a orientação absoluta do segundo segmento no referencial global. Esse princípio generaliza-se: para uma cadeia de $n$ segmentos, o ângulo absoluto da $i$-ésima junta é $\Theta_i = \sum_{k=1}^{i} \theta_k$.

O simulador a seguir permite verificar essa relação geometricamente, variando $\theta_1$ e $\theta_2$ de forma contínua.

In [1]:
from IPython.display import HTML
HTML("""<div style=\"border:1px solid #d0d0d0;border-radius:6px;padding:1.25rem;margin:1.5rem 0;background:#fafafa;\">\n<p style=\"font-size:0.8rem;font-weight:600;letter-spacing:0.06em;text-transform:uppercase;color:#555;margin:0 0 0.75rem;\">Simulador — Cinemática direta</p>\n<canvas id=\"legCanvas\" width=\"540\" height=\"300\" style=\"display:block;margin:0 auto;border:1px solid #e8e8e8;border-radius:4px;background:#fff;\"></canvas>\n<div style=\"max-width:540px;margin:0.75rem auto 0;\">\n  <div style=\"display:flex;align-items:center;gap:10px;margin-bottom:6px;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:100px;color:#333;\">θ₁ (1ª junta)</span>\n    <input type=\"range\" id=\"th1\" min=\"-70\" max=\"70\" value=\"30\" style=\"flex:1;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:36px;color:#1a7a50;\" id=\"th1val\">30°</span>\n  </div>\n  <div style=\"display:flex;align-items:center;gap:10px;margin-bottom:6px;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:100px;color:#333;\">θ₂ (2ª junta)</span>\n    <input type=\"range\" id=\"th2\" min=\"-70\" max=\"70\" value=\"20\" style=\"flex:1;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:36px;color:#2563a8;\" id=\"th2val\">20°</span>\n  </div>\n  <p style=\"font-size:12px;font-family:monospace;color:#666;text-align:center;margin-top:6px;\" id=\"footpos\">Calculando...</p>\n</div>\n</div>\n<script>\n(function(){\n  var cv=document.getElementById('legCanvas');\n  var ctx=cv.getContext('2d');\n  var L=110,cx=270,cy=120;\n  function draw(){\n    var t1=parseFloat(document.getElementById('th1').value)*Math.PI/180;\n    var t2=parseFloat(document.getElementById('th2').value)*Math.PI/180;\n    document.getElementById('th1val').textContent=Math.round(t1*180/Math.PI)+'°';\n    document.getElementById('th2val').textContent=Math.round(t2*180/Math.PI)+'°';\n    var x1=cx+L*Math.cos(t1),y1=cy-L*Math.sin(t1);\n    var x2=x1+L*Math.cos(t1+t2),y2=y1-L*Math.sin(t1+t2);\n    ctx.clearRect(0,0,540,300);\n    ctx.strokeStyle='#e8e8e8';ctx.lineWidth=0.5;\n    for(var i=0;i<=540;i+=40){ctx.beginPath();ctx.moveTo(i,0);ctx.lineTo(i,300);ctx.stroke();}\n    for(var j=0;j<=300;j+=40){ctx.beginPath();ctx.moveTo(0,j);ctx.lineTo(540,j);ctx.stroke();}\n    ctx.strokeStyle='#bbb';ctx.lineWidth=1;\n    ctx.beginPath();ctx.moveTo(0,cy);ctx.lineTo(540,cy);ctx.stroke();\n    ctx.beginPath();ctx.moveTo(cx,0);ctx.lineTo(cx,300);ctx.stroke();\n    ctx.strokeStyle='#1a7a50';ctx.lineWidth=5;ctx.lineCap='round';\n    ctx.beginPath();ctx.moveTo(cx,cy);ctx.lineTo(x1,y1);ctx.stroke();\n    ctx.strokeStyle='#2563a8';ctx.lineWidth=5;\n    ctx.beginPath();ctx.moveTo(x1,y1);ctx.lineTo(x2,y2);ctx.stroke();\n    [[cx,cy,'#1a7a50',7],[x1,y1,'#888',5],[x2,y2,'#c0392b',7]].forEach(function(p){\n      ctx.beginPath();ctx.arc(p[0],p[1],p[3],0,2*Math.PI);ctx.fillStyle=p[2];ctx.fill();\n    });\n    ctx.font='12px monospace';ctx.fillStyle='#1a7a50';\n    ctx.fillText('θ₁='+Math.round(t1*180/Math.PI)+'°',cx+8,cy-10);\n    ctx.fillStyle='#2563a8';\n    ctx.fillText('θ₂='+Math.round(t2*180/Math.PI)+'°',x1+8,y1-8);\n    ctx.fillStyle='#c0392b';ctx.font='bold 12px monospace';\n    ctx.fillText('pé (x₂, y₂)',x2+8,y2+4);\n    var rx=((x2-cx)/L).toFixed(3),ry=((-(y2-cy))/L).toFixed(3);\n    document.getElementById('footpos').textContent='x₂ = '+rx+' L    y₂ = '+ry+' L   (normalizado por L)';\n  }\n  document.getElementById('th1').addEventListener('input',draw);\n  document.getElementById('th2').addEventListener('input',draw);\n  draw();\n})();\n</script>""")

### 1.4 Implementação no MATLAB — cinemática estática

O código a seguir calcula e plota a configuração da perna para ângulos fixos. Altere `theta1` e `theta2` e observe o deslocamento do pé.

```{note}
**Figura a incluir:** execute o código abaixo, descomente a linha `exportgraphics` e insira a imagem gerada no espaço reservado após o bloco de código.
```

```matlab
clc; clear; close all;

% Comprimento dos segmentos [cm]
L1 = 5;
L2 = 5;

% Angulos das juntas [graus] — altere estes valores para explorar
theta1 = deg2rad(30);
theta2 = deg2rad(20);

% Junta intermediaria (joelho)
x1 = L1 * cos(theta1);
y1 = L1 * sin(theta1);

% Extremidade (pe)
x2 = x1 + L2 * cos(theta1 + theta2);
y2 = y1 + L2 * sin(theta1 + theta2);

figure('Position', [100 100 520 420]);

plot([0 x1], [0 y1], '-o', 'LineWidth', 4, 'Color', [0.10 0.48 0.31], ...
     'MarkerSize', 8, 'MarkerFaceColor', [0.10 0.48 0.31]);
hold on;
plot([x1 x2], [y1 y2], '-o', 'LineWidth', 4, 'Color', [0.15 0.39 0.66], ...
     'MarkerSize', 8, 'MarkerFaceColor', [0.15 0.39 0.66]);
plot(x2, y2, 'o', 'MarkerSize', 12, 'MarkerFaceColor', [0.75 0.22 0.17], ...
     'MarkerEdgeColor', 'none');

grid on; axis equal; axis([-11 11 -11 11]);
xlabel('x [cm]'); ylabel('y [cm]');
title(sprintf('Cinematica direta — pe em (%.2f, %.2f) cm', x2, y2));
legend('Segmento 1 (L_1)', 'Segmento 2 (L_2)', 'Pe', 'Location', 'northwest');

% Para exportar: descomente a linha abaixo
% exportgraphics(gcf, 'fig01_cinematica_estatica.png', 'Resolution', 150);
```

```{figure} fig01_cinematica_estatica.png
:name: fig-cinematica-estatica
:width: 80%
:align: center

Configuração geométrica da perna com θ₁ = 30° e θ₂ = 20°. O segmento verde representa o primeiro elo e o azul, o segundo. O marcador vermelho indica a posição do pé calculada pelas equações de cinemática direta.
```

---

## 2. Movimento no tempo — modelagem da caminhada

### 2.1 Funções senoidais como modelos de movimento oscilatório

A caminhada não é uma sequência de posições estáticas. Os ângulos das juntas variam continuamente no tempo de forma periódica. Em engenharia, movimentos periódicos são modelados por funções senoidais:

$$
\theta(t) = A \sin(\omega t + \varphi)
$$

| Parâmetro | Unidade | Significado físico | Efeito na caminhada |
|---|---|---|---|
| $A$ | graus | Amplitude — deflexão angular máxima | Comprimento do passo |
| $\omega$ | rad/s | Frequência angular | Cadência |
| $\varphi$ | rad | Fase inicial | Sincronismo entre as pernas |

O mesmo modelo aparece em dinâmica estrutural (vibrações de pontes), hidráulica (oscilação de marés) e engenharia sísmica (acelerograma simplificado). A senoide é um modelo universal para fenômenos oscilatórios.

### 2.2 O papel do desfasamento de fase

As duas pernas do Otto oscilam com a mesma amplitude e frequência, mas com fases distintas. Um desfasamento de $\varphi = \pi/2$ entre as duas pernas significa que, enquanto uma atinge o ângulo máximo, a outra está na posição neutra. Esse desfasamento é o que produz o padrão de passo coordenado.

Em estruturas, desfasamentos entre componentes vibratórios levam a batimentos e amplificações indesejadas. No bípede, o desfasamento é deliberadamente controlado para produzir movimento estável.

O simulador a seguir permite explorar o efeito de $A$, $\omega$ e $\varphi$ sobre o movimento resultante.

In [1]:
from IPython.display import HTML
HTML("""<div style=\"border:1px solid #d0d0d0;border-radius:6px;padding:1.25rem;margin:1.5rem 0;background:#fafafa;\">\n<p style=\"font-size:0.8rem;font-weight:600;letter-spacing:0.06em;text-transform:uppercase;color:#555;margin:0 0 0.75rem;\">Simulador — Movimento senoidal</p>\n<canvas id=\"wCanvas\" width=\"540\" height=\"180\" style=\"display:block;margin:0 auto;border:1px solid #e8e8e8;border-radius:4px;background:#fff;\"></canvas>\n<div style=\"max-width:540px;margin:0.75rem auto 0;\">\n  <div style=\"display:flex;align-items:center;gap:10px;margin-bottom:6px;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:110px;color:#333;\">A (amplitude)</span>\n    <input type=\"range\" id=\"wAmp\" min=\"5\" max=\"50\" value=\"20\" style=\"flex:1;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:44px;color:#1a7a50;\" id=\"wAmpV\">20°</span>\n  </div>\n  <div style=\"display:flex;align-items:center;gap:10px;margin-bottom:6px;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:110px;color:#333;\">ω (freq. angular)</span>\n    <input type=\"range\" id=\"wFreq\" min=\"1\" max=\"8\" value=\"2\" step=\"0.5\" style=\"flex:1;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:44px;color:#2563a8;\" id=\"wFreqV\">2 rad/s</span>\n  </div>\n  <div style=\"display:flex;align-items:center;gap:10px;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:110px;color:#333;\">φ (fase)</span>\n    <input type=\"range\" id=\"wPhi\" min=\"0\" max=\"628\" value=\"0\" style=\"flex:1;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:44px;color:#7a3a8a;\" id=\"wPhiV\">0 rad</span>\n  </div>\n</div>\n</div>\n<script>\n(function(){\n  var cv=document.getElementById('wCanvas');\n  var ctx=cv.getContext('2d');\n  var W=540,H=180,cy=H/2;\n  var t0=0;\n  function draw(){\n    var A=parseFloat(document.getElementById('wAmp').value);\n    var w=parseFloat(document.getElementById('wFreq').value);\n    var phi=parseFloat(document.getElementById('wPhi').value)/100;\n    document.getElementById('wAmpV').textContent=A+'°';\n    document.getElementById('wFreqV').textContent=w+' rad/s';\n    document.getElementById('wPhiV').textContent=(phi/Math.PI).toFixed(2)+'π rad';\n    ctx.clearRect(0,0,W,H);\n    ctx.strokeStyle='#ebebeb';ctx.lineWidth=0.5;\n    for(var i=0;i<W;i+=40){ctx.beginPath();ctx.moveTo(i,0);ctx.lineTo(i,H);ctx.stroke();}\n    for(var j=0;j<H;j+=30){ctx.beginPath();ctx.moveTo(0,j);ctx.lineTo(W,j);ctx.stroke();}\n    ctx.strokeStyle='#bbb';ctx.lineWidth=1;\n    ctx.beginPath();ctx.moveTo(0,cy);ctx.lineTo(W,cy);ctx.stroke();\n    ctx.beginPath();ctx.strokeStyle='#1a7a50';ctx.lineWidth=2;\n    for(var px=0;px<W;px++){\n      var t=(px/W)*12+t0;\n      var v=A*Math.sin(w*t+phi);\n      var py=cy-(v/55)*(H/2-8);\n      if(px===0)ctx.moveTo(px,py);else ctx.lineTo(px,py);\n    }\n    ctx.stroke();\n    var tc=t0+3,vc=A*Math.sin(w*tc+phi);\n    var pyc=cy-(vc/55)*(H/2-8),pxc=(3/12)*W;\n    ctx.beginPath();ctx.arc(pxc,pyc,5,0,2*Math.PI);ctx.fillStyle='#c0392b';ctx.fill();\n    ctx.font='11px monospace';ctx.fillStyle='#555';\n    ctx.fillText('θ(t) = '+A+'·sin('+w+'t + φ)',8,14);\n    t0+=0.018;\n    requestAnimationFrame(draw);\n  }\n  draw();\n})();\n</script>""")

### 2.3 Implementação no MATLAB — posição, velocidade e aceleração angular

```{note}
**Figura a incluir:** execute o código, descomente `exportgraphics` e insira a figura abaixo.
```

```matlab
clc; clear; close all;

A   = 20;       % amplitude [graus]
w   = 2;        % frequencia angular [rad/s]
phi = pi/4;     % fase [rad]

t = linspace(0, 10, 1000);

theta   = A * sin(w*t + phi);
omega_t = gradient(theta, t);
alpha_t = gradient(omega_t, t);

figure('Position', [100 100 640 400]);
plot(t, theta,   'LineWidth', 2, 'Color', [0.10 0.48 0.31]); hold on;
plot(t, omega_t, 'LineWidth', 2, 'Color', [0.15 0.39 0.66]);
plot(t, alpha_t, 'LineWidth', 2, 'Color', [0.75 0.22 0.17]);
yline(0, 'k--', 'LineWidth', 0.5);
grid on;
xlabel('Tempo [s]'); ylabel('Magnitude');
title(sprintf('Posicao, velocidade e aceleracao angular  (A=%g graus, omega=%g rad/s)', A, w));
legend('theta(t) [graus]', 'omega(t) = dtheta/dt [graus/s]', ...
       'alpha(t) = d2theta/dt2 [graus/s2]', 'Location', 'northeast');

% exportgraphics(gcf, 'fig02_senoide_cinematica.png', 'Resolution', 150);
```

```{figure} fig02_senoide_cinematica.png
:name: fig-senoide
:width: 85%
:align: center

Posição, velocidade e aceleração angular de uma junta com θ(t) = 20°·sin(2t + π/4). A amplitude da aceleração angular é Aω² = 80 °/s², evidenciando o crescimento quadrático da demanda de torque com a frequência.
```

### 2.4 Dinâmica rotacional e torque

O servomotor precisa produzir torque suficiente para vencer a inércia rotacional do segmento. A equação fundamental é:

$$
\tau = I\,\alpha
$$

onde $\tau$ é o torque [N·m], $I$ é o momento de inércia [kg·m²] e $\alpha$ é a aceleração angular [rad/s²].

Para o movimento $\theta(t) = A\sin(\omega t)$, a aceleração angular é $\alpha(t) = -A\omega^2\sin(\omega t)$. O torque máximo necessário é:

$$
\tau_{\max} = I\,A\omega^2
$$

Dobrar a frequência de caminhada multiplica a demanda de torque por quatro. Esse é o limite físico que impede o robô de caminhar arbitrariamente rápido.

A gravidade impõe um torque adicional sobre cada segmento:

$$
\tau_g = m\,g\,r\,\cos(\theta)
$$

onde $r$ é a distância do centro de massa ao eixo de rotação. O valor máximo ocorre para $\theta = 0$ (segmento horizontal), exatamente como no cálculo de momento fletor máximo em uma viga bi-apoiada com carga central.

---

## 3. Controle em malha fechada

### 3.1 A necessidade do controle

Um servomotor real não responde como um dispositivo ideal. Ao receber uma referência angular, o servo apresenta:

- **Atraso de resposta:** intervalo de tempo entre o comando e o início do movimento;
- **Sobressinal (*overshoot*):** ultrapassagem da posição desejada antes da estabilização;
- **Erro em regime estacionário:** diferença residual entre a posição final e a referência, causada por atrito ou não linearidades.

Sem correção, esses erros se acumulam ao longo da caminhada e o robô perde equilíbrio. O controlador em malha fechada soluciona esse problema comparando continuamente a posição real com a referência e calculando a ação corretiva necessária.

### 3.2 Estrutura da malha de controle

```
r(t) ──►[Σ]──► C(s) ──► G(s) ──► y(t)
          │-                        │
          └──────── H(s) ◄──────────┘
```

- $r(t)$: referência — ângulo desejado;
- $y(t)$: saída — ângulo real medido pelo sensor;
- $e(t) = r(t) - y(t)$: sinal de erro;
- $C(s)$: controlador (PID);
- $G(s)$: planta (modelo do servo);
- $H(s)$: sensor (assume-se unitário, $H(s) = 1$).

O simulador abaixo permite observar como os ganhos do PID afetam a resposta do servo a uma referência senoidal.

In [1]:
from IPython.display import HTML
HTML("""<div style=\"border:1px solid #d0d0d0;border-radius:6px;padding:1.25rem;margin:1.5rem 0;background:#fafafa;\">\n<p style=\"font-size:0.8rem;font-weight:600;letter-spacing:0.06em;text-transform:uppercase;color:#555;margin:0 0 0.75rem;\">Simulador — Resposta do servo com controlador PID (modelo discreto simplificado)</p>\n<canvas id=\"pidCanvas\" width=\"540\" height=\"200\" style=\"display:block;margin:0 auto;border:1px solid #e8e8e8;border-radius:4px;background:#fff;\"></canvas>\n<div style=\"max-width:540px;margin:0.75rem auto 0;\">\n  <div style=\"display:flex;align-items:center;gap:10px;margin-bottom:6px;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:60px;color:#333;\">Kp</span>\n    <input type=\"range\" id=\"pKp\" min=\"0.1\" max=\"5\" value=\"1.2\" step=\"0.1\" style=\"flex:1;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:36px;color:#1a7a50;\" id=\"pKpV\">1.2</span>\n  </div>\n  <div style=\"display:flex;align-items:center;gap:10px;margin-bottom:6px;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:60px;color:#333;\">Ki</span>\n    <input type=\"range\" id=\"pKi\" min=\"0\" max=\"2\" value=\"0.3\" step=\"0.05\" style=\"flex:1;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:36px;color:#2563a8;\" id=\"pKiV\">0.30</span>\n  </div>\n  <div style=\"display:flex;align-items:center;gap:10px;margin-bottom:6px;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:60px;color:#333;\">Kd</span>\n    <input type=\"range\" id=\"pKd\" min=\"0\" max=\"1\" value=\"0.15\" step=\"0.05\" style=\"flex:1;\">\n    <span style=\"font-size:13px;font-family:monospace;min-width:36px;color:#7a3a8a;\" id=\"pKdV\">0.15</span>\n  </div>\n  <p style=\"font-size:11px;color:#888;margin-top:4px;font-family:monospace;\">Verde = referência r(t) &nbsp;|&nbsp; Azul = saída y(t) &nbsp;|&nbsp; Vermelho = erro e(t)</p>\n</div>\n</div>\n<script>\n(function(){\n  var cv=document.getElementById('pidCanvas');\n  var ctx=cv.getContext('2d');\n  var W=540,H=200,cy=H/2;\n  var t0=0;\n  function simulate(Kp,Ki,Kd){\n    var dt=0.02,N=Math.floor(12/dt);\n    var ref=[],out=[],err=[];\n    var y=0,yi=0,ep=0;\n    for(var i=0;i<N;i++){\n      var t=i*dt,r=20*Math.sin(2*t),e=r-y;\n      yi+=e*dt;\n      var de=(e-ep)/dt,u=Kp*e+Ki*yi+Kd*de;\n      y+=u*dt*3;y*=0.92;\n      ref.push(r);out.push(y);err.push(e);ep=e;\n    }\n    return {ref:ref,out:out,err:err,N:N};\n  }\n  function draw(){\n    var Kp=parseFloat(document.getElementById('pKp').value);\n    var Ki=parseFloat(document.getElementById('pKi').value);\n    var Kd=parseFloat(document.getElementById('pKd').value);\n    document.getElementById('pKpV').textContent=Kp.toFixed(1);\n    document.getElementById('pKiV').textContent=Ki.toFixed(2);\n    document.getElementById('pKdV').textContent=Kd.toFixed(2);\n    var d=simulate(Kp,Ki,Kd);\n    ctx.clearRect(0,0,W,H);\n    ctx.strokeStyle='#ebebeb';ctx.lineWidth=0.5;\n    for(var i=0;i<W;i+=40){ctx.beginPath();ctx.moveTo(i,0);ctx.lineTo(i,H);ctx.stroke();}\n    for(var j=0;j<H;j+=30){ctx.beginPath();ctx.moveTo(0,j);ctx.lineTo(W,j);ctx.stroke();}\n    ctx.strokeStyle='#bbb';ctx.lineWidth=1;\n    ctx.beginPath();ctx.moveTo(0,cy);ctx.lineTo(W,cy);ctx.stroke();\n    var sc=H/2/28,shift=Math.floor(t0*50)%d.N;\n    function drawCurve(arr,color){\n      ctx.beginPath();ctx.strokeStyle=color;ctx.lineWidth=1.5;\n      for(var p=0;p<W;p++){\n        var idx=(Math.floor(p/W*d.N)+shift)%d.N;\n        var py=cy-arr[idx]*sc;\n        if(p===0)ctx.moveTo(p,py);else ctx.lineTo(p,py);\n      }\n      ctx.stroke();\n    }\n    drawCurve(d.ref,'#1a7a50');\n    drawCurve(d.out,'#2563a8');\n    drawCurve(d.err,'#c0392b');\n    t0+=0.025;\n    requestAnimationFrame(draw);\n  }\n  draw();\n  ['pKp','pKi','pKd'].forEach(function(id){\n    document.getElementById(id).addEventListener('input',draw);\n  });\n})();\n</script>""")

### 3.3 O controlador PID

A lei de controle do PID é:

$$
u(t) = K_p\,e(t) + K_i\int_0^t e(\tau)\,d\tau + K_d\,\frac{de(t)}{dt}
$$

Cada termo tem função física distinta:

**Termo proporcional ($K_p$).** Produz ação corretiva proporcional ao erro atual. Um ganho elevado reduz o erro rapidamente, mas pode induzir oscilações persistentes.

**Termo integral ($K_i$).** Acumula o erro ao longo do tempo e elimina o erro em regime estacionário. É análogo a um ajuste progressivo de projeto em resposta a deformações observadas — a correção cresce até que o desvio seja nulo.

**Termo derivativo ($K_d$).** Reage à taxa de variação do erro. Se o erro diminui rapidamente, o termo derivativo reduz a ação corretiva para evitar o *overshoot*. Funcionalmente, é equivalente a um amortecedor viscoso em sistemas estruturais: resiste às variações rápidas e dissipa energia oscilante.

### 3.4 Função de transferência do servo

O comportamento dinâmico do servo é representado por uma função de transferência de primeira ordem:

$$
G(s) = \frac{1}{\tau s + 1}
$$

Para $\tau = 0{,}1$ s, o servo leva aproximadamente $5\tau = 0{,}5$ s para atingir 99% do valor final em resposta a um degrau unitário. Este modelo captura o atraso de resposta essencial sem requerer identificação paramétrica completa do motor. A função de transferência em malha fechada com controlador PID é:

$$
T(s) = \frac{C(s)\,G(s)}{1 + C(s)\,G(s)}
$$

---

## 4. Implementação no MATLAB e Simulink

### 4.1 Cinemática dinâmica — animação da perna

O código a seguir combina os resultados das Seções 1 e 2: a cinemática direta é avaliada quadro a quadro com ângulos variando senoidalmente no tempo.

```{note}
**Figura a incluir:** capture um quadro representativo em $t = 2$ s (comentários no código indicam como fazer) e insira abaixo.
```

```matlab
clc; clear; close all;

L1 = 5; L2 = 5;
t = linspace(0, 10, 300);
trajX = zeros(1, length(t));
trajY = zeros(1, length(t));

figure('Position', [100 100 480 480]);

for k = 1:length(t)
    theta1 = deg2rad(20 * sin(t(k)));
    theta2 = deg2rad(15 * sin(t(k) + pi/2));

    x1 = L1 * cos(theta1);
    y1 = L1 * sin(theta1);
    x2 = x1 + L2 * cos(theta1 + theta2);
    y2 = y1 + L2 * sin(theta1 + theta2);
    trajX(k) = x2; trajY(k) = y2;

    clf;
    plot(trajX(1:k), trajY(1:k), ':', 'Color', [0.6 0.6 0.6], 'LineWidth', 1);
    hold on;
    plot([0 x1], [0 y1], '-o', 'LineWidth', 4, 'Color', [0.10 0.48 0.31], ...
         'MarkerSize', 7, 'MarkerFaceColor', [0.10 0.48 0.31]);
    plot([x1 x2], [y1 y2], '-o', 'LineWidth', 4, 'Color', [0.15 0.39 0.66], ...
         'MarkerSize', 7, 'MarkerFaceColor', [0.15 0.39 0.66]);
    plot(x2, y2, 'o', 'MarkerSize', 10, ...
         'MarkerFaceColor', [0.75 0.22 0.17], 'MarkerEdgeColor', 'none');
    grid on; axis equal;
    xlim([-11 11]); ylim([-11 11]);
    xlabel('x [cm]'); ylabel('y [cm]');
    title(sprintf('t = %.2f s', t(k)));
    drawnow;
end

% Para capturar um quadro em t = 2 s:
% k_alvo = find(t >= 2.0, 1);
% ... calcule e plote apenas para k = k_alvo ...
% exportgraphics(gcf, 'fig03_cinematica_dinamica.png', 'Resolution', 150);
```

```{figure} fig03_cinematica_dinamica.png
:name: fig-cinematica-dinamica
:width: 70%
:align: center

Configuração instantânea da perna em t = 2 s com θ₁(t) = 20°·sin(t) e θ₂(t) = 15°·sin(t + π/2). A linha pontilhada representa a trajetória acumulada do pé desde t = 0.
```

### 4.2 Modelo Simulink — malha de controle

Monte no Simulink a seguinte cadeia de blocos:

| Bloco | Parâmetros | Descrição |
|---|---|---|
| **Sine Wave** | Amplitude: 20, Frequência: 2 rad/s, Fase: 0 | Referência $r(t)$ |
| **Sum** | Sinais: `+−` | Calcula $e(t) = r(t) - y(t)$ |
| **PID Controller** | Kp = 1.2, Ki = 0.3, Kd = 0.15 | Controlador |
| **Transfer Fcn** | Numerador: `[1]`, Denominador: `[0.1 1]` | Modelo do servo $G(s)$ |
| **Scope** | 2 entradas | Visualização de $r(t)$ e $y(t)$ |

O canal de realimentação conecta a saída da Transfer Fcn à entrada negativa do bloco Sum.

```{note}
**Figura a incluir:** capture a tela do Scope após 10 s de simulação e insira abaixo. Identifique o atraso de fase, o overshoot e o comportamento em regime estacionário.
```

```{figure} fig04_scope_simulink.png
:name: fig-scope
:width: 85%
:align: center

Saída do Scope: referência r(t) (verde) e resposta do servo y(t) (azul) para Kp = 1.2, Ki = 0.3, Kd = 0.15. Observe o atraso de fase entre as curvas e o overshoot nos picos.
```

### 4.3 Análise do sistema de controle com o Control System Toolbox

O código a seguir calcula a resposta ao degrau do sistema em malha fechada e extrai indicadores quantitativos de desempenho. Esses indicadores serão utilizados nas próximas aulas para ajuste formal dos ganhos.

```{note}
**Figura a incluir:** execute o código e insira a figura da resposta ao degrau abaixo.
```

```matlab
clc; clear; close all;

% Funcao de transferencia do servo: G(s) = 1 / (0.1s + 1)
G = tf(1, [0.1 1]);

% Ganhos do controlador PID
Kp = 1.2;
Ki = 0.3;
Kd = 0.15;

C = pid(Kp, Ki, Kd);

% Sistema em malha fechada
T = feedback(C * G, 1);

% Resposta ao degrau
figure('Position', [100 100 620 380]);
[y, t] = step(T, linspace(0, 3, 500));
plot(t, y, 'LineWidth', 2, 'Color', [0.15 0.39 0.66]); hold on;
yline(1, '--', 'Referencia', 'LineWidth', 1, 'Color', [0.10 0.48 0.31], ...
      'LabelHorizontalAlignment', 'left');
grid on;
xlabel('Tempo [s]'); ylabel('Posicao normalizada');
title('Resposta ao degrau — sistema em malha fechada com PID');

info = stepinfo(T);
fprintf('Tempo de subida:     %.3f s\n', info.RiseTime);
fprintf('Tempo de acomoda:    %.3f s\n', info.SettlingTime);
fprintf('Overshoot:           %.2f %%\n', info.Overshoot);

% exportgraphics(gcf, 'fig05_resposta_degrau.png', 'Resolution', 150);
```

```{figure} fig05_resposta_degrau.png
:name: fig-degrau
:width: 85%
:align: center

Resposta ao degrau do sistema servo + PID em malha fechada (Kp = 1.2, Ki = 0.3, Kd = 0.15). Os indicadores de desempenho — tempo de subida, tempo de acomodação e overshoot — são calculados pela função stepinfo do Control System Toolbox.
```

---

## 5. Síntese e perspectivas

### 5.1 Conceitos abordados

Esta aula introduziu três camadas de descrição matemática de um sistema robótico articulado:

1. **Cinemática direta.** A posição do pé é determinada pelos ângulos das juntas por transformações trigonométricas encadeadas. Para uma cadeia de $n$ segmentos, a posição final é $\mathbf{p} = \sum_{i=1}^{n} L_i [\cos\Theta_i,\, \sin\Theta_i]^\top$, onde $\Theta_i = \sum_{k=1}^{i}\theta_k$.

2. **Dinâmica do movimento.** O ângulo de cada junta varia segundo $\theta(t) = A\sin(\omega t + \varphi)$. O torque dinâmico máximo necessário cresce com $\omega^2$, impondo limites à cadência de caminhada.

3. **Controle em malha fechada.** O controlador PID minimiza continuamente o erro $e(t) = r(t) - y(t)$ por meio de três ações complementares: proporcional, integral e derivativa.

### 5.2 Roteiro das próximas aulas

| Aula | Conteúdo | Ferramentas |
|---|---|---|
| 2 | Identificação experimental do servo | System Identification Toolbox |
| 3 | Ajuste formal de ganhos PID | Control System Toolbox |
| 4 | Modelagem cinemática completa do bípede | Robotics System Toolbox |
| 5 | Lógica de caminhada e máquina de estados | Stateflow |

---

## Exercícios

**1.** Calcule analiticamente a posição do pé do Otto para $\theta_1 = 45°$, $\theta_2 = -30°$, $L_1 = L_2 = 5$ cm. Verifique o resultado com o simulador interativo da Seção 1.

**2.** No código da Seção 2.3, altere $\omega$ de 2 para 4 rad/s. Qual é a variação percentual na amplitude da aceleração angular? Confirme algebricamente com a expressão $\alpha_{\max} = A\omega^2$.

**3.** No código da Seção 4.3, configure $K_p = 5$, $K_i = 0$, $K_d = 0$ e registre os valores de `stepinfo`. Em seguida, acrescente $K_d = 0.5$ e compare. Explique fisicamente a redução do *overshoot*.

**4.** O torque máximo do micro servo SG90 é 1,8 kgf·cm = 0,177 N·m. Modele um segmento de perna como uma haste uniforme de 5 cm e 10 g, com $I = mL^2/3$. Para $A = 20°$ e $\omega = 4$ rad/s, o torque dinâmico máximo excede a capacidade do servo?

**5.** Em um sistema de amortecimento ativo estrutural, identifique os equivalentes de $r(t)$, $y(t)$, $e(t)$ e $u(t)$ do diagrama de malha fechada apresentado na Seção 3.2. Discuta o papel de $K_d$ como amortecedor ativo.